<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

folder = '/content/drive/MyDrive/aml-project'

for f in os.listdir(folder):
    size_mb = os.path.getsize(os.path.join(folder, f)) / 1e6
    print(f"{f}  ({size_mb:.1f} MB)")

SAML-D.csv  (996.2 MB)


In [3]:
import pandas as pd

file = '/content/drive/MyDrive/aml-project/SAML-D.csv'

peek = pd.read_csv(file, nrows=1000)

print("Shape of peek:", peek.shape)
print("\nColumns:")
for col in peek.columns:
    print(f"  {col:<25} {peek[col].dtype}")

Shape of peek: (1000, 12)

Columns:
  Time                      object
  Date                      object
  Sender_account            int64
  Receiver_account          int64
  Amount                    float64
  Payment_currency          object
  Received_currency         object
  Sender_bank_location      object
  Receiver_bank_location    object
  Payment_type              object
  Is_laundering             int64
  Laundering_type           object


In [4]:
with open(file) as f:
    total_rows = sum(1 for _ in f) - 1

print(f"Total rows: {total_rows:,}")

Total rows: 9,504,852


In [5]:
counts = pd.read_csv(file, usecols=['Is_laundering'])['Is_laundering'].value_counts()

clean, criminal = counts[0], counts[1]
total = clean + criminal

print(f"Clean:     {clean:>10,}  ({clean/total:.4%})")
print(f"Laundering:{criminal:>10,}  ({criminal/total:.4%})")
print(f"\nRatio: 1 in {round(total/criminal):,} transactions")

Clean:      9,494,979  (99.8961%)
Laundering:     9,873  (0.1039%)

Ratio: 1 in 963 transactions


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(peek.head(5))
print("\n--- Amount summary ---")
print(peek['Amount'].describe())

       Time        Date  Sender_account  Receiver_account    Amount Payment_currency Received_currency Sender_bank_location Receiver_bank_location  Payment_type  Is_laundering       Laundering_type
0  10:35:19  2022-10-07      8724731955        2769355426   1459.15        UK pounds         UK pounds                   UK                     UK  Cash Deposit              0  Normal_Cash_Deposits
1  10:35:20  2022-10-07      1491989064        8401255335   6019.64        UK pounds            Dirham                   UK                    UAE  Cross-border              0        Normal_Fan_Out
2  10:35:20  2022-10-07       287305149        4404767002  14328.44        UK pounds         UK pounds                   UK                     UK        Cheque              0  Normal_Small_Fan_Out
3  10:35:21  2022-10-07      5376652437        9600420220  11895.00        UK pounds         UK pounds                   UK                     UK           ACH              0         Normal_Fan_In
4  10:35:2

In [7]:
lt = pd.read_csv(file, usecols=['Laundering_type', 'Is_laundering'])

print("=== CRIMINAL patterns ===")
print(lt[lt['Is_laundering'] == 1]['Laundering_type'].value_counts())

print("\n=== NORMAL patterns ===")
print(lt[lt['Is_laundering'] == 0]['Laundering_type'].value_counts())

=== CRIMINAL patterns ===
Laundering_type
Structuring             1870
Cash_Withdrawal         1334
Deposit-Send             945
Smurfing                 932
Layered_Fan_In           656
Layered_Fan_Out          529
Stacked Bipartite        506
Behavioural_Change_1     394
Bipartite                383
Cycle                    382
Fan_In                   364
Gather-Scatter           354
Behavioural_Change_2     345
Scatter-Gather           338
Single_large             250
Fan_Out                  237
Over-Invoicing            54
Name: count, dtype: int64

=== NORMAL patterns ===
Laundering_type
Normal_Small_Fan_Out      3477717
Normal_Fan_Out            2302220
Normal_Fan_In             2104285
Normal_Group               528351
Normal_Cash_Withdrawal     305031
Normal_Cash_Deposits       223801
Normal_Periodical          210526
Normal_Plus_Mutual         155041
Normal_Mutual              125335
Normal_Foward               42031
Normal_single_large         20641
Name: count, dtype: int6

In [8]:
cat = pd.read_csv(file, usecols=['Payment_type', 'Sender_bank_location', 'Date'])

print("Payment types:")
print(cat['Payment_type'].value_counts())

print(f"\nSender locations: {cat['Sender_bank_location'].nunique()} distinct")
print(cat['Sender_bank_location'].value_counts().head(8))

print(f"\nDate range: {cat['Date'].min()} to {cat['Date'].max()}")
print(f"Distinct days: {cat['Date'].nunique()}")

Payment types:
Payment_type
Credit card        2012909
Debit card         2012103
Cheque             2011419
ACH                2008807
Cross-border        933931
Cash Withdrawal     300477
Cash Deposit        225206
Name: count, dtype: int64

Sender locations: 18 distinct
Sender_bank_location
UK             9183088
Turkey           20902
Switzerland      20503
Pakistan         20346
UAE              20081
Nigeria          20027
Spain            19391
Germany          19259
Name: count, dtype: int64

Date range: 2022-10-07 to 2023-08-23
Distinct days: 321


In [9]:
import numpy as np

cols = ['Time','Date','Sender_account','Receiver_account','Amount',
        'Payment_currency','Received_currency','Sender_bank_location',
        'Receiver_bank_location','Payment_type','Is_laundering','Laundering_type']

chunks_crime, chunks_clean = [], []
rng = np.random.default_rng(42)
keep_rate = 1_500_000 / 9_494_979

for chunk in pd.read_csv(file, usecols=cols, chunksize=500_000):
    chunks_crime.append(chunk[chunk['Is_laundering'] == 1])
    clean = chunk[chunk['Is_laundering'] == 0]
    mask = rng.random(len(clean)) < keep_rate
    chunks_clean.append(clean[mask])

df = pd.concat(chunks_crime + chunks_clean, ignore_index=True)
df = df.sort_values(['Date','Time']).reset_index(drop=True)

print(f"Sample rows:  {len(df):,}")
print(f"  criminal:   {df['Is_laundering'].sum():,}")
print(f"  clean:      {(df['Is_laundering']==0).sum():,}")
print(f"Memory:       {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

Sample rows:  1,509,825
  criminal:   9,873
  clean:      1,499,952
Memory:       0.74 GB


In [10]:
out = '/content/drive/MyDrive/aml-project/sample.parquet'
df.to_parquet(out, index=False)

size = os.path.getsize(out) / 1e6
print(f"Saved: {size:.0f} MB")

Saved: 29 MB


In [11]:
crime = df[df['Is_laundering'] == 1]
clean = df[df['Is_laundering'] == 0]

print("=== Amount: criminal vs clean ===")
comparison = pd.DataFrame({
    'criminal': crime['Amount'].describe(),
    'clean': clean['Amount'].describe()
})
print(comparison.round(2))

=== Amount: criminal vs clean ===
          criminal       clean
count      9873.00  1499952.00
mean      40587.67     8733.58
std      419181.13    21640.35
min          15.82        5.32
25%        2723.79     2143.16
50%        5322.79     6121.54
75%        9789.67    10463.20
max    12618498.40   999962.19


In [12]:
bins = [0, 1000, 2000, 3000, 5000, 7500, 9000, 9500, 10000,
        15000, 25000, 50000, 1e9]

crime_binned = pd.cut(crime['Amount'], bins=bins).value_counts().sort_index()
clean_binned = pd.cut(clean['Amount'], bins=bins).value_counts().sort_index()

result = pd.DataFrame({'criminal': crime_binned, 'clean': clean_binned})
result['crime_rate_%'] = (result['criminal'] / (result['criminal'] + result['clean']) * 100).round(3)
print(result)

                         criminal   clean  crime_rate_%
Amount                                                 
(0.0, 1000.0]                1437  236092         0.605
(1000.0, 2000.0]              490  124840         0.391
(2000.0, 3000.0]              799   84680         0.935
(3000.0, 5000.0]             1953  164832         1.171
(5000.0, 7500.0]             1721  285384         0.599
(7500.0, 9000.0]              721  129398         0.554
(9000.0, 9500.0]              185   37166         0.495
(9500.0, 10000.0]             171   33591         0.506
(10000.0, 15000.0]            916  216684         0.421
(15000.0, 25000.0]            655  133048         0.490
(25000.0, 50000.0]            423   41815         1.001
(50000.0, 1000000000.0]       402   12422         3.135


In [13]:
by_type = crime.groupby('Laundering_type')['Amount'].agg(
    n='count', median='median', p25=lambda s: s.quantile(.25),
    p75=lambda s: s.quantile(.75), max='max'
).round(0).sort_values('n', ascending=False)

print(by_type)

                         n     median        p25        p75         max
Laundering_type                                                        
Structuring           1870     4800.0     3701.0     6375.0     12454.0
Cash_Withdrawal       1334      139.0       82.0      214.0       350.0
Deposit-Send           945     6332.0     3436.0    12014.0     43910.0
Smurfing               932     2629.0     1936.0     3428.0      6985.0
Layered_Fan_In         656    10726.0     7218.0    18987.0    117015.0
Layered_Fan_Out        529     6730.0     4730.0    10367.0     37748.0
Stacked Bipartite      506     7883.0     5858.0    11835.0     44651.0
Behavioural_Change_1   394     3118.0     1560.0     6682.0     19276.0
Bipartite              383     8901.0     6398.0    14164.0     56710.0
Cycle                  382    21495.0    10708.0    36985.0    134975.0
Fan_In                 364     8552.0     5195.0    15481.0     64993.0
Gather-Scatter         354     8510.0     5609.0    13805.0     

In [17]:
crime_senders = crime['Sender_account'].value_counts()
clean_senders = clean['Sender_account'].value_counts()

print(f"Criminal txns from {len(crime_senders):,} distinct senders")
print(f"Median txns per criminal sender: {crime_senders.median():.0f}")
print(f"Max: {crime_senders.max():,}")
print(f"\nTop 10 criminal senders:")
print(crime_senders.head(10))
print(f"\nMedian txns per clean sender: {clean_senders.median():.0f}")

Criminal txns from 4,950 distinct senders
Median txns per criminal sender: 1
Max: 37

Top 10 criminal senders:
Sender_account
4159678387    37
9710838491    31
4503049074    30
9330449479    29
9285172899    28
3990750333    28
9772022469    26
2488893433    26
5262095561    26
7147448786    26
Name: count, dtype: int64

Median txns per clean sender: 2


In [18]:
sender_stats = df.groupby('Sender_account').agg(
    n_txns=('Is_laundering', 'size'),
    n_criminal=('Is_laundering', 'sum')
)
bad = sender_stats[sender_stats['n_criminal'] > 0].copy()
bad['pct_criminal'] = (bad['n_criminal'] / bad['n_txns'] * 100).round(1)

print(f"Accounts with >=1 criminal txn: {len(bad)}")
print(bad.sort_values('n_txns', ascending=False).head(15))
print(f"\nMedian % of a bad account's txns that are criminal: {bad['pct_criminal'].median():.1f}%")

Accounts with >=1 criminal txn: 4950
                n_txns  n_criminal  pct_criminal
Sender_account                                  
361478571          141           7           5.0
3747015869         135           9           6.7
8192587178         135           7           5.2
8535122087         133           5           3.8
9416949366         132           6           4.5
6174306361         130           7           5.4
4808614002         130           8           6.2
8600542721         129           8           6.2
7332435745         129           6           4.7
8974559268         127           5           3.9
403526894          127           8           6.3
5460360634         126           6           4.8
4728492996         126           7           5.6
4569841157         126           7           5.6
6123762963         125           8           6.4

Median % of a bad account's txns that are criminal: 6.5%


In [19]:
recv = df.groupby('Receiver_account')['Is_laundering'].agg(['size','sum'])
recv_bad = recv[recv['sum'] > 0]

print(f"Distinct receivers of criminal txns: {len(recv_bad)}")
print(f"Distinct receivers overall: {df['Receiver_account'].nunique():,}")

overlap = set(bad.index) & set(recv_bad.index)
print(f"Accounts that both send AND receive criminal: {len(overlap)}")

Distinct receivers of criminal txns: 4074
Distinct receivers overall: 480,006
Accounts that both send AND receive criminal: 1122


In [20]:
sender_all = df.groupby('Sender_account').agg(
    n_txns=('Is_laundering','size'),
    n_criminal=('Is_laundering','sum'),
    total_amt=('Amount','sum'),
    med_amt=('Amount','median'),
    n_receivers=('Receiver_account','nunique')
)
sender_all['is_bad'] = (sender_all['n_criminal'] > 0).astype(int)

print(f"Total senders: {len(sender_all):,}  |  bad: {sender_all['is_bad'].sum():,}")
print("\n=== Median profile ===")
print(sender_all.groupby('is_bad')[
    ['n_txns','total_amt','med_amt','n_receivers']].median().round(0))

Total senders: 223,816  |  bad: 4,950

=== Median profile ===
        n_txns  total_amt  med_amt  n_receivers
is_bad                                         
0          2.0    17665.0   8062.0          1.0
1         23.0   169518.0   4892.0         11.0


In [21]:
df['hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

hourly = df.groupby('hour')['Is_laundering'].agg(['size','sum'])
hourly['rate_%'] = (hourly['sum']/hourly['size']*100).round(3)
print(hourly)

       size  sum  rate_%
hour                    
0     19243  136   0.707
1     19471  122   0.627
2     19443  138   0.710
3     19262  138   0.716
4     19153  126   0.658
5     19719  140   0.710
6     19233  146   0.759
7     22709  222   0.978
8     85313  632   0.741
9     85667  527   0.615
10    85946  589   0.685
11    85704  599   0.699
12    85920  577   0.672
13    86030  601   0.699
14    85838  566   0.659
15    85387  516   0.604
16    85830  556   0.648
17    85599  579   0.676
18    82630  494   0.598
19    82500  497   0.602
20    82438  456   0.553
21    81967  516   0.630
22    82481  501   0.607
23    82342  499   0.606


In [22]:
df = df.drop(columns=['hour'])
df.to_parquet('/content/drive/MyDrive/aml-project/sample.parquet', index=False)

findings = {
    'total_rows_full_dataset': 9_504_852,
    'sample_rows': len(df),
    'criminal_full': 9873,
    'base_rate_full_%': 0.1039,
    'base_rate_sample_%': round(df['Is_laundering'].mean()*100, 4),
    'criminal_senders': 4950,
    'criminal_receivers': 4074,
    'pass_through_accounts': 1122,
    'median_pct_criminal_per_bad_account': 6.5,
    'date_from': df['Date'].min(),
    'date_to': df['Date'].max(),
}
pd.Series(findings).to_csv('/content/drive/MyDrive/aml-project/phase1_findings.csv')
for k, v in findings.items():
    print(f"{k:<38} {v}")

total_rows_full_dataset                9504852
sample_rows                            1509825
criminal_full                          9873
base_rate_full_%                       0.1039
base_rate_sample_%                     0.6539
criminal_senders                       4950
criminal_receivers                     4074
pass_through_accounts                  1122
median_pct_criminal_per_bad_account    6.5
date_from                              2022-10-07
date_to                                2023-08-23
